# BazaarTalks — test in Google Colab

> A research platform for **multi-market equity screening, backtesting, and factor

## What this notebook does
Clones the repo, installs its dependencies, and runs a **code smoke test** (byte-compiles every module and imports the entry points) so you can confirm the code is sound before running it for real.

## Requirements

**Python packages:** `pandas==2.3.3, numpy==2.0.2, pyarrow==21.0.0          # parquet read/write (dataset outputs), openpyxl==3.1.5          # styled Excel workbook export, yfinance==1.2.0          # OHLC + fundamentals, all markets, requests==2.32.5, nsepython==2.97          # India: NSE universe + quotes, bseindia==1.1            # India: BSE listings, kabupy                   # Japan: JPX/TSE universe (unpinned upstream), pykrx                    # South Korea: KRX/Naver (unpinned upstream), scikit-learn==1.6.1      # Ridge/Linear regression directional signal, scipy==1.13.1, cassandra-driver==3.29.3 # required: OHLC cache in Cassandra (market_store.py), confluent-kafka==2.14.2  # required: CDC publish (market_store.py) + work-queue (stream_pipeline.py), apache-flink==2.3.0      # OPTIONAL: flink_cdc_consumer.py / flink_screens.py do the same jobs, duckdb==1.4.5           # DuckDB SQL layer over parquets + attached SQLite (warehouse.py), playwright==1.48.0      # trendlyne_session.py — Trendlyne is a JS-rendered SPA, needs a real`

**Entry points detected:**
- `accumulation_screener.py`
- `alerts.py`
- `apply_costs.py`
- `benchmark.py`
- `build_india_seed.py`
- `corporate_actions.py`
- `crowding.py`
- `darvas_volume.py`
- `dashboard.py`
- `data_quality.py`
- `data_sources.py`
- `dvm_composite.py`

## Before you run for real — caveats
- **Needs credentials/secrets** (API keys, .env, or mailer/broker logins) — set these in the Colab session before running the relevant scripts; they are gitignored and not in the repo.
- **Needs data files** (DuckDB/parquet/cache). Some are committed (incl. via Git LFS — run `!git lfs pull` in Colab); others are generated by the collectors or fetched live.
- Repo uses **Git LFS** — the clone cell runs `git lfs pull` to fetch large data files.

*Generated testing scaffold — edit freely.*


In [ ]:
# Clone the repo
!git clone --depth 1 https://github.com/herrrickshaw/BazaarTalks.git
%cd BazaarTalks
!git lfs pull || true

In [ ]:
# Install dependencies
import os
if os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt
elif os.path.exists('pyproject.toml'):
    !pip install -q -e .
else:
    print('No requirements.txt / pyproject.toml — nothing to install')

In [ ]:
# Code smoke test: byte-compile every module (syntax/def-level check)
import subprocess, sys, glob
pyfiles = glob.glob('**/*.py', recursive=True)
print(f'Compiling {len(pyfiles)} Python files...')
r = subprocess.run([sys.executable, '-m', 'py_compile', *pyfiles],
                   text=True, capture_output=True)
print('OK — all modules compile' if r.returncode == 0
      else 'Compile errors:\n' + r.stderr)

In [ ]:
# List entry points and their docstrings — how to run each
import ast, glob
for f in sorted(glob.glob('*.py') + glob.glob('collectors/*.py')
                + glob.glob('scripts/*.py') + glob.glob('src/*.py')):
    try:
        doc = ast.get_docstring(ast.parse(open(f).read())) or ''
    except Exception:
        doc = ''
    print(f'\n=== {f} ===')
    print(doc.strip()[:400])
    print(f'Run with:  !python {f} --help')